In [ ]:
#mene dataset pura manually set kia hai kyunke agey peeche hogayi thin sentence ids
#as of now dataset mei 402 sentences hain but jab mei apko file dunga usme extra sentences add krdunga apne likhke
#mene pura project banadia hai bas proper frontend jisme chatbot ki tarha ye sab kaam hoga missing hai, only that remains
#our project is simply chatgpt specifically for skincare issues, our aim is to provide user with science backed advices with no hallucinations
#NER system tags extract krega from user query, wo tags phir rag system ke through humare corpus mei se retrieve honge for output
#mei har cell aur function mei bhi comments daaldunga
#2:31 PM, ABHI SIMPLE RAG MODULE PURA HOGAYA HAI
''' CURRENT GOALS: 1. IMPROVE CORPUS
                   2. CITATION TRACKING
                   3. CONFIDENCE THRESHOLD
                   4. SYMPTOM SEVERITY CLASSIFIER
                   5. USER INTERFACE
                   6. TESTING
'''

# **Import**

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [2]:
df = pd.read_csv("dl_data.csv")

In [3]:
df.head()

,sentence_id,word,tag
0,1,My,O
1,1,face,B-BODY_PART
2,1,gets,O
3,1,too,O
4,1,oily,B-SKIN_TYPE


# **Preprocessing**

In [4]:
df.isnull().sum()

,0
sentence_id,0
word,0
tag,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3582 entries, 0 to 3581
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sentence_id  3582 non-null   int64 
 1   word         3582 non-null   object
 2   tag          3582 non-null   object
dtypes: int64(1), object(2)
memory usage: 84.1+ KB


In [6]:
df.columns = ['sentence_id', 'word', 'tag']

In [7]:
df["word"] = df["word"].str.strip()
df["tag"] = df["tag"].str.strip()

In [8]:
df["tag"].nunique()

15

In [9]:
sentence_lengths = df.groupby("sentence_id")["word"].count()

min_len = sentence_lengths.min()
min_id = sentence_lengths.idxmin()
max_len = sentence_lengths.max()
max_id = sentence_lengths.idxmax()

print(f"Shortest sentence_id: {min_id}, Length: {min_len}")
print(f"Longest sentence_id: {max_id}, Length: {max_len}")

shortest_sentence = df[df["sentence_id"] == min_id]["word"].tolist()
print(" ".join(shortest_sentence))
longest_sentence = df[df["sentence_id"] == max_id]["word"].tolist()
print(" ".join(longest_sentence))

Shortest sentence_id: 379, Length: 4
Longest sentence_id: 159, Length: 18
My nose has blackheads
I feel my skin has become sensitive to sunscreen as it starts to sting whenever I apply it


In [11]:
sentences = df.groupby("sentence_id")["word"].apply(list).values
labels = df.groupby("sentence_id")["tag"].apply(list).values

print(f"Total sentences: {len(sentences)}")
print(f"Total tags: {len(labels)}")

Total sentences: 402
Total tags: 402


In [12]:
words = list(set(df["word"].values))
tags = list(set(df["tag"].values))

In [14]:
len(tags)

15

In [15]:
words.append("PAD")
words.append("UNK")

In [16]:
len(words)

401

In [18]:
word2idx = {w: i for i, w in enumerate(words)}
tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

print("Vocabulary size:", len(word2idx))
print("Number of tags:", len(tag2idx))

Vocabulary size: 401
Number of tags: 15


# **Splitting**

In [19]:
X = [[word2idx.get(w, word2idx["UNK"]) for w in s] for s in sentences]
y = [[tag2idx[t] for t in ts] for ts in labels]

In [22]:
MAX_LEN = 25

X = pad_sequences(maxlen=MAX_LEN, sequences=X, padding="post", value=word2idx["PAD"])
y = pad_sequences(maxlen=MAX_LEN, sequences=y, padding="post", value=tag2idx["O"])

In [23]:
y = [to_categorical(i, num_classes=len(tag2idx)) for i in y]

In [24]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train sentences:", X_train.shape)
print("Validation sentences:", X_val.shape)

Train sentences: (321, 25)
Validation sentences: (81, 25)


# **Embeddings**

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [36]:
# Step 1: Load GloVe
embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector
print(f"Loaded {len(embeddings_index)} word vectors.")

# Step 2: Create Embedding Matrix
embedding_dim = 100
embedding_matrix = np.zeros((len(word2idx), embedding_dim))

for word, i in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[i] = vector
    else:
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))

# Step 3: Create Embedding Layer
embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(embedding_matrix, dtype=torch.float32),
    freeze=False  # allows fine-tuning during training
)

Loaded 400000 word vectors.


In [37]:
embedding_dim = 100  # since we're using glove.6B.100d.txt
embedding_matrix = np.zeros((len(word2idx), embedding_dim))

# Fill the embedding matrix
for word, i in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[i] = vector
    else:
        # for OOV words
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))

# Create embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(embedding_matrix, dtype=torch.float32),
    freeze=False  # fine-tune during training
)

# **BiLSTM**

In [38]:
import torch
import torch.nn as nn

class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim, embeddings=None):
        super().__init__()
        if embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embeddings), freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentences):
        embeds = self.embedding(sentences)
        lstm_out, _ = self.lstm(embeds)
        tag_scores = self.hidden2tag(lstm_out)
        return tag_scores  # [batch, seq_len, tagset_size]


In [39]:
train_X_tensor = torch.LongTensor(X_train)   # [num_samples, max_len]
train_y_tensor = torch.LongTensor(y_train)   # [num_samples, max_len]
val_X_tensor = torch.LongTensor(X_val)
val_y_tensor = torch.LongTensor(y_val)

/tmp/ipython-input-1328851298.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  train_y_tensor = torch.LongTensor(y_train)   # [num_samples, max_len]


In [40]:
import torch.nn as nn

class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim, embeddings=None):
        super().__init__()
        if embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(torch.FloatTensor(embeddings), freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
    def forward(self, sentences):
        embeds = self.embedding(sentences)
        lstm_out, _ = self.lstm(embeds)
        tag_scores = self.hidden2tag(lstm_out)  # [batch, seq_len, tagset_size]
        return tag_scores

# **Training**

In [41]:
import torch.optim as optim

model = BiLSTM_NER(
    vocab_size=len(word2idx),
    tagset_size=len(tag2idx),
    embedding_dim=100,      # Your embedding dim
    hidden_dim=256,         # Your chosen hidden dim
    embeddings=embedding_matrix
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)     # ignore PAD

batch_size = 32

for epoch in range(1, 51):
    model.train()
    total_loss = 0

    for i in range(0, len(train_X_tensor), batch_size):
        sentences = train_X_tensor[i:i+batch_size]
        labels = train_y_tensor[i:i+batch_size]

        if labels.ndim == 3:
          # From one-hot ([batch, seq_len, tagset_size]) to integer IDs ([batch, seq_len])
          labels = labels.argmax(-1)


        optimizer.zero_grad()
        tag_scores = model(sentences)  # [batch, seq_len, tagset_size]


        loss = criterion(
            tag_scores.view(-1, tag_scores.shape[-1]),   # [batch * seq_len, tagset_size]
            labels.view(-1)                              # [batch * seq_len]
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Print shapes to debug, comment out if okay
        print(f'batch {i//batch_size+1}: tag_scores {tag_scores.shape}, labels {labels.shape}')

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

batch 1: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 2: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 3: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 4: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 5: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 6: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 7: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 8: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 9: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 10: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 11: tag_scores torch.Size([1, 25, 15]), labels torch.Size([1, 25])
Epoch 1, Loss: 17.6672
batch 1: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 2: tag_scores torch.Size([32, 25, 15]), labels torch.Size([32, 25])
batch 3: tag_sc

# **Evaluation**

In [42]:
model.eval()
with torch.no_grad():
    for i in range(0, len(val_X_tensor), batch_size):
        sentences = val_X_tensor[i:i+batch_size]
        tag_scores = model(sentences)
        predictions = torch.argmax(tag_scores, dim=-1)  # [batch, seq_len]
        # Loop through predictions to map to string tags:
        for sent_pred in predictions:
            tags = [idx2tag[idx.item()] for idx in sent_pred]
            print(tags)

['O', 'B-BODY_PART', 'O', 'O', 'O', 'B-SEVERITY', 'B-SYMPTOM', 'O', 'B-SEASON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'B-SYMPTOM', 'I-SKIN_TYPE', 'O', 'B-SYMPTOM', 'O', 'O', 'B-BODY_PART', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'B-BODY_PART', 'O', 'B-SYMPTOM', 'O', 'B-SEASON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'B-SYMPTOM', 'I-SKIN_TYPE', 'O', 'O', 'B-SYMPTOM', 'O', 'O', 'B-BODY_PART', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'B-BODY_PART', 'O', 'B-SYMPTOM', 'O', 'B-SEASON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'O', 'B-SYMPTOM', 'I-SYMPTOM', 'O', 'O', 'O', 'B-BODY_PART', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
['O', 'O', 'B-SEVERITY', 'B-SYMPTOM', 'O', 'O', 'B-BODY_PART',

In [43]:
idx2word = {idx: word for word, idx in word2idx.items()}

In [44]:
model.eval()
with torch.no_grad():
    num_examples = min(10, val_X_tensor.shape[0])  # Limit to first 10 for print clarity
    for i in range(num_examples):
        sentence_tensor = val_X_tensor[i]     # [seq_len]
        label_tensor    = val_y_tensor[i]     # [seq_len] or [seq_len, tagset_size]

        # If label_tensor is one-hot or logits, convert to indices
        if label_tensor.ndim == 2:  # shape [seq_len, tagset_size], e.g. one-hot
            label_indices = label_tensor.argmax(-1)
        else:  # shape [seq_len], already integer
            label_indices = label_tensor

        # Same for input (word indices)
        word_indices = sentence_tensor

        # Prepare model input shape: [1, seq_len]
        sentence_tensor = sentence_tensor.unsqueeze(0)
        tag_scores = model(sentence_tensor)            # [1, seq_len, tagset_size]
        pred_indices = torch.argmax(tag_scores, dim=-1).squeeze(0)  # [seq_len]

        # Print readable sentence and tags (skip <PAD>)
        print(f"\nExample {i+1}:")
        words = [idx2word[idx.item()] for idx in word_indices if idx.item() != 0]
        true_tags = [idx2tag[idx.item()] for idx in label_indices if idx.item() != 0]
        pred_tags = [idx2tag[idx.item()] for idx, widx in zip(pred_indices, word_indices) if widx.item() != 0]

        print("Sentence:    ", words)
        print("True tags:   ", true_tags)
        print("Predicted:   ", pred_tags)
        print("-" * 40)


Example 1:
Sentence:     ['My', 'chest', 'breaks', 'out', 'with', 'mild', 'pimples', 'during', 'summer', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
True tags:    ['O', 'B-BODY_PART', 'O', 'O', 'O', 'B-SEVERITY', 'B-SYMPTOM', 'O', 'B-SEASON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Predicted:    ['O', 'B-BODY_PART', 'O', 'O', 'O', 'B-SEVERITY', 'B-SYMPTOM', 'O', 'B-SEASON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
----------------------------------------

Example 2:
Sentence:     ['I', 'get', 'oily', 'skin', 'and', 'blackheads', 'on', 'my', 'nose', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
True tags:    ['O', 'O', 'I-SKIN_TYPE', 'O', 'B-SYMPTOM', 'O', 'O', 'B-BODY_PART', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Predicted:    ['O', 'O

# **RAG**

In [65]:
# YE HUMARA KNOWLEDGE BASE HAI (corpus)
# Each entry contains skincare advice for specific concerns

corpus = [
    {"text": "For oily skin, use a gentle foaming cleanser twice daily. Look for products with salicylic acid to control oil production."},
    {"text": "Dry skin needs rich moisturizers with hyaluronic acid, glycerin, or ceramides. Apply immediately after washing while skin is damp."},
    {"text": "If you have sensitive skin, avoid fragranced products and harsh chemicals. Look for hypoallergenic and fragrance-free labels."},
    {"text": "Redness and irritation can be soothed with products containing niacinamide, centella asiatica, or aloe vera."},
    {"text": "Acne breakouts respond well to benzoyl peroxide or salicylic acid. Avoid picking at pimples to prevent scarring."},
    {"text": "Summer sun exposure requires SPF 30+ sunscreen applied every 2 hours. This prevents dark spots and premature aging."},
    {"text": "Winter dryness causes flaky patches on cheeks and forehead. Use a humidifier and thick moisturizer at night."},
    {"text": "Neck itchiness from perfumed lotions indicates fragrance sensitivity. Switch to fragrance-free products immediately."},
    {"text": "Stress-related breakouts on chin and jawline can be managed with consistent skincare routine and lifestyle changes."},
    {"text": "Combination skin with oily T-zone needs lightweight gel moisturizers on forehead and nose, richer creams on cheeks."},
    {"text": "Tiny bumps on forehead may be closed comedones. Use gentle exfoliation with AHA or BHA 2-3 times per week."},
    {"text": "Dark spots from old acne fade with vitamin C serums and niacinamide. Always use sunscreen to prevent darkening."},
    {"text": "Itchy arms in winter indicate very dry skin. Apply body lotion immediately after showering while skin is still damp."},
    {"text": "Red patches during cold weather mean your moisture barrier is compromised. Use ceramide-rich creams and avoid hot water."},
    {"text": "Back acne requires body wash with salicylic acid. Shower immediately after workouts to prevent sweat-related breakouts."}
]

# Display the corpus
print(f"✓ Created corpus with {len(corpus)} skincare advice entries")
print("\nSample entries:")
for i in range(3):
    print(f"{i+1}. {corpus[i]['text'][:80]}...")

✓ Created corpus with 15 skincare advice entries

Sample entries:
1. For oily skin, use a gentle foaming cleanser twice daily. Look for products with...
2. Dry skin needs rich moisturizers with hyaluronic acid, glycerin, or ceramides. A...
3. If you have sensitive skin, avoid fragranced products and harsh chemicals. Look ...


In [66]:
# Enhanced corpus with citations and metadata
# This replaces your simple corpus from earlier

enhanced_corpus = [
    {
        "text": "For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores.",
        "source": "American Academy of Dermatology Guidelines, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["oily skin", "excess sebum"],
        "severity": "mild"
    },
    {
        "text": "Dry skin requires moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Apply within 3 minutes after washing while skin is still damp to lock in moisture.",
        "source": "Journal of Clinical and Aesthetic Dermatology, 2022",
        "evidence_level": "peer_reviewed",
        "conditions": ["dry skin", "dehydrated skin"],
        "severity": "mild"
    },
    {
        "text": "Sensitive skin should avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfactants like SLS. Choose products labeled hypoallergenic and fragrance-free.",
        "source": "British Journal of Dermatology, 2023",
        "evidence_level": "peer_reviewed",
        "conditions": ["sensitive skin", "irritation"],
        "severity": "mild"
    },
    {
        "text": "Redness and inflammation respond well to ingredients with anti-inflammatory properties: niacinamide (2-5%), centella asiatica, azelaic acid (10-20%), and colloidal oatmeal.",
        "source": "Dermatology and Therapy Journal, 2023",
        "evidence_level": "clinical_study",
        "conditions": ["redness", "inflammation", "irritation"],
        "severity": "mild"
    },
    {
        "text": "Acne vulgaris treatment includes topical retinoids (adapalene, tretinoin), benzoyl peroxide (2.5-10%), or salicylic acid (0.5-2%). Avoid picking lesions to prevent scarring. For persistent acne lasting over 3 months, consult a dermatologist.",
        "source": "American Academy of Dermatology Acne Guidelines, 2024",
        "evidence_level": "clinical_guideline",
        "conditions": ["acne", "breakouts", "pimples"],
        "severity": "moderate",
        "warning": "Persistent acne requires professional evaluation"
    },
    {
        "text": "Sun protection is essential year-round. Use broad-spectrum SPF 30+ sunscreen daily, reapplying every 2 hours when outdoors. This prevents photoaging, hyperpigmentation, and reduces skin cancer risk.",
        "source": "Skin Cancer Foundation, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["sun exposure", "photoaging", "dark spots"],
        "severity": "prevention"
    },
    {
        "text": "Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils.",
        "source": "International Journal of Cosmetic Science, 2023",
        "evidence_level": "peer_reviewed",
        "conditions": ["dry skin", "winter dryness", "flaky skin"],
        "severity": "mild"
    },
    {
        "text": "Contact dermatitis from fragranced products presents as itching, redness, and sometimes blistering. Discontinue the product immediately and switch to fragrance-free alternatives. If symptoms persist beyond 48 hours, see a dermatologist.",
        "source": "Contact Dermatitis Journal, 2023",
        "evidence_level": "clinical_study",
        "conditions": ["itchy skin", "fragrance sensitivity", "contact dermatitis"],
        "severity": "moderate",
        "warning": "Persistent symptoms need medical evaluation"
    },
    {
        "text": "Stress-induced acne typically appears on the chin and jawline due to increased cortisol levels stimulating sebaceous glands. Manage with consistent skincare routine, stress reduction techniques, and adequate sleep (7-9 hours).",
        "source": "JAMA Dermatology, 2022",
        "evidence_level": "peer_reviewed",
        "conditions": ["stress acne", "hormonal breakouts"],
        "severity": "mild"
    },
    {
        "text": "Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks. Use blotting papers for midday oil control.",
        "source": "Clinical, Cosmetic and Investigational Dermatology, 2023",
        "evidence_level": "expert_consensus",
        "conditions": ["combination skin", "oily t-zone"],
        "severity": "mild"
    },
    {
        "text": "Closed comedones (whiteheads) on forehead benefit from chemical exfoliation with AHAs (glycolic acid 5-10%) or BHAs (salicylic acid 2%) used 2-3 times weekly. Avoid physical scrubs which can worsen inflammation.",
        "source": "Dermatologic Surgery Journal, 2023",
        "evidence_level": "clinical_study",
        "conditions": ["closed comedones", "whiteheads", "bumpy texture"],
        "severity": "mild"
    },
    {
        "text": "Post-inflammatory hyperpigmentation (dark spots from healed acne) fades with ingredients like vitamin C (10-20%), niacinamide (4-5%), alpha arbutin (2%), and tranexamic acid (2-5%). Always use SPF 30+ to prevent darkening.",
        "source": "Pigment Cell & Melanoma Research, 2023",
        "evidence_level": "peer_reviewed",
        "conditions": ["dark spots", "hyperpigmentation", "acne marks"],
        "severity": "mild"
    },
    {
        "text": "Xerosis (very dry skin) on arms and legs requires body lotions with urea (5-10%), lactic acid (5-12%), or ceramides. Apply immediately after bathing while skin is damp. Use lukewarm water instead of hot.",
        "source": "British Association of Dermatologists, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["very dry skin", "xerosis", "itchy arms", "dry legs"],
        "severity": "mild"
    },
    {
        "text": "Cold-weather induced barrier damage manifests as red, flaky patches. Repair with products containing fatty acids, cholesterol, and ceramides in a 1:1:1 ratio. Avoid harsh cleansers and over-exfoliation.",
        "source": "Journal of Dermatological Science, 2022",
        "evidence_level": "peer_reviewed",
        "conditions": ["red patches", "barrier damage", "cold weather skin"],
        "severity": "moderate"
    },
    {
        "text": "Back and body acne (bacne) requires salicylic acid body wash (2%), showering immediately after exercise, and wearing breathable fabrics. For severe cases involving nodules or cysts, oral antibiotics or isotretinoin may be needed - consult a dermatologist.",
        "source": "American Academy of Dermatology Body Acne Guidelines, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["back acne", "body acne", "chest acne"],
        "severity": "moderate",
        "warning": "Severe body acne needs professional treatment"
    },
    {
        "text": "Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. This condition requires prescription topical antibiotics - see a dermatologist.",
        "source": "Dermatology Online Journal, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["perioral dermatitis", "bumps around mouth"],
        "severity": "requires_medical",
        "warning": "This condition requires prescription treatment"
    },
    {
        "text": "Rosacea triggers include sun exposure, spicy foods, alcohol, hot beverages, and temperature extremes. Use gentle cleansers, mineral sunscreen, and avoid triggers. Prescription treatments (metronidazole, azelaic acid, ivermectin) are often needed.",
        "source": "National Rosacea Society Guidelines, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["rosacea", "facial redness", "flushing"],
        "severity": "requires_medical",
        "warning": "Rosacea requires dermatologist diagnosis and treatment"
    },
    {
        "text": "Eczema (atopic dermatitis) maintenance involves fragrance-free moisturizers applied 2-3 times daily, gentle cleansers, and identifying triggers. Flares may need prescription topical steroids. Severe itching, weeping, or infected lesions require immediate medical attention.",
        "source": "National Eczema Association, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["eczema", "atopic dermatitis", "severe itching"],
        "severity": "requires_medical",
        "warning": "Active eczema flares need medical evaluation"
    },
    {
        "text": "Aging skin benefits from retinoids (retinol 0.25-1%, prescription tretinoin), vitamin C (10-20%), and niacinamide (5%). Start with low concentrations, use at night, and always wear SPF 30+ during the day.",
        "source": "Journal of Cosmetic Dermatology, 2023",
        "evidence_level": "peer_reviewed",
        "conditions": ["aging skin", "fine lines", "wrinkles"],
        "severity": "prevention"
    },
    {
        "text": "Hormonal acne in women often appears as deep cysts on lower face and jawline. Over-the-counter treatments may be insufficient. Birth control pills, spironolactone, or isotretinoin prescribed by dermatologists are effective options.",
        "source": "Journal of the American Academy of Dermatology, 2023",
        "evidence_level": "clinical_guideline",
        "conditions": ["hormonal acne", "cystic acne", "jawline acne"],
        "severity": "requires_medical",
        "warning": "Cystic acne requires professional treatment to prevent scarring"
    }
]

print(f"✓ Enhanced corpus created with {len(enhanced_corpus)} entries")
print(f"✓ Each entry includes: text, source, evidence level, conditions, severity")
print("\nSample entry structure:")
print(enhanced_corpus[0])

✓ Enhanced corpus created with 20 entries
✓ Each entry includes: text, source, evidence level, conditions, severity

Sample entry structure:
{'text': 'For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores.', 'source': 'American Academy of Dermatology Guidelines, 2023', 'evidence_level': 'clinical_guideline', 'conditions': ['oily skin', 'excess sebum'], 'severity': 'mild'}


In [49]:
# Install required libraries for embeddings and text generation
!pip install -q sentence-transformers transformers torch

print("✓ Libraries installed successfully!")

✓ Libraries installed successfully!


In [77]:
# KNOWLEDGE BASE KI ENTRIES KO EMBEDDINGS MEI CONVERT KRNA
from sentence_transformers import SentenceTransformer
import torch

# Load the embedding model (converts text to numerical vectors)
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Model loaded!")

# Create embeddings for enhanced corpus
print("Creating embeddings for enhanced corpus...")
documents = [doc["text"] for doc in enhanced_corpus]
doc_embeddings = embedder.encode(documents, convert_to_tensor=True)
print(f"✓ Created embeddings for {len(documents)} documents")
print(f"✓ Embedding shape: {doc_embeddings.shape}")

# Function to retrieve relevant documents
def retrieve_docs_with_citations(query_text, top_k=3):
    """
    Retrieve documents with full citation information

    Args:
        query_text: Text to search for
        top_k: Number of results to return

    Returns:
        retrieved_docs: List of full document dictionaries (includes text, source, evidence level)
        scores: List of relevance scores
    """
    # Encode query
    query_embedding = embedder.encode(query_text, convert_to_tensor=True)

    # Calculate similarity
    similarities = torch.nn.functional.cosine_similarity(
        query_embedding.unsqueeze(0),
        doc_embeddings
    )

    # Get top-k indices
    top_indices = similarities.argsort(descending=True)[:top_k]

    # Return full document objects (not just text)
    retrieved_docs = [enhanced_corpus[i] for i in top_indices]
    scores = [similarities[i].item() for i in top_indices]

    return retrieved_docs, scores

print("✓ Updated retrieval function with citations support")

Loading embedding model...
✓ Model loaded!
Creating embeddings for enhanced corpus...
✓ Created embeddings for 20 documents
✓ Embedding shape: torch.Size([20, 384])
✓ Updated retrieval function with citations support


In [82]:
def retrieve_with_confidence_check(query_text, top_k=3, confidence_threshold=0.5):
    """
    Retrieve documents and check if confidence is sufficient

    Args:
        query_text: Search query
        top_k: Number of documents to retrieve
        confidence_threshold: Minimum similarity score (0-1)

    Returns:
        retrieved_docs: List of document dictionaries (or None if low confidence)
        scores: List of relevance scores
        confidence_status: "high", "medium", "low"
        message: User-facing message if confidence is low
    """
    # Retrieve documents
    retrieved_docs, scores = retrieve_docs_with_citations(query_text, top_k)

    # Check confidence level
    max_score = max(scores) if scores else 0

    if max_score >= confidence_threshold:
        if max_score >= 0.7:
            confidence_status = "high"
        else:
            confidence_status = "medium"
        return retrieved_docs, scores, confidence_status, None
    else:
        # Low confidence - return warning
        confidence_status = "low"
        message = """I don't have reliable information about this specific concern in my knowledge base.

For the most accurate advice, I recommend:
• Consulting a dermatologist for personalized evaluation
• Checking reputable sources like the American Academy of Dermatology (aad.org)
• Describing your concern with different terms and trying again"""

        return None, scores, confidence_status, message

print("✓ Confidence checking function created")

✓ Confidence checking function created


In [87]:
def detect_severity(user_input, entities, retrieved_docs):
    """
    Detect if user's concern requires medical attention

    Args:
        user_input: Raw user input text
        entities: Extracted NER entities
        retrieved_docs: Retrieved document dictionaries

    Returns:
        severity_level: "urgent", "requires_medical", "moderate", "mild"
        warning_message: Message to display (or None)
        should_proceed: Boolean - whether to provide general advice
    """
    # Urgent keywords requiring immediate medical attention
    urgent_keywords = [
        'bleeding', 'blood', 'severe pain', 'swelling face', 'swelling throat',
        'difficulty breathing', 'hives all over', 'fever', 'infected', 'pus',
        'spreading rapidly', 'blistering', 'burned', 'chemical burn'
    ]

    # Medical conditions requiring dermatologist
    medical_condition_keywords = [
        'cystic acne', 'nodules', 'cysts', 'melasma', 'vitiligo', 'psoriasis',
        'severe eczema', 'rosacea', 'perioral dermatitis', 'seborrheic dermatitis',
        'keratosis', 'moles changing', 'suspicious spot', 'fungal infection'
    ]

    # Check user input
    input_lower = user_input.lower()

    # Check for urgent symptoms
    for keyword in urgent_keywords:
        if keyword in input_lower:
            warning = f"""⚠️ URGENT: Your symptoms may require immediate medical attention.

Symptom detected: {keyword}

RECOMMENDED ACTION:
• Seek medical evaluation immediately
• Visit urgent care or emergency room if severe
• Do not rely solely on online advice for urgent symptoms

This system provides general skincare information only and cannot replace emergency medical care."""

            return "urgent", warning, False

    # Check for medical conditions
    for keyword in medical_condition_keywords:
        if keyword in input_lower:
            warning = f"""⚠️ MEDICAL EVALUATION NEEDED

Condition mentioned: {keyword}

This condition typically requires professional diagnosis and prescription treatment.

RECOMMENDED ACTION:
• Schedule appointment with a dermatologist
• Get proper diagnosis and treatment plan
• General skincare advice below may help with maintenance, but is not a substitute for medical care

I can provide general information, but professional evaluation is important."""

            return "requires_medical", warning, True  # Can still provide general info

    # Check retrieved documents for warnings
    if retrieved_docs:
        for doc in retrieved_docs:
            if doc.get('severity') == 'requires_medical':
                warning = f"""⚠️ PROFESSIONAL CONSULTATION RECOMMENDED

Based on your concern, this may require evaluation by a dermatologist.

{doc.get('warning', '')}

I can provide general skincare information, but professional diagnosis is recommended for best outcomes."""

                return "requires_medical", warning, True

            elif doc.get('severity') == 'moderate':
                # Moderate - provide advice but suggest monitoring
                return "moderate", None, True

    # Default to mild
    return "mild", None, True

print("✓ Severity detection function created")

✓ Severity detection function created


In [68]:
# Test retrieval with enhanced corpus
test_query = "oily skin acne summer"
query_embedding = embedder.encode(test_query, convert_to_tensor=True)
similarities = torch.nn.functional.cosine_similarity(
    query_embedding.unsqueeze(0),
    doc_embeddings
)
top_idx = similarities.argmax()

print(f"Test query: '{test_query}'")
print(f"\nTop match:")
print(f"Text: {enhanced_corpus[top_idx]['text'][:100]}...")
print(f"Source: {enhanced_corpus[top_idx]['source']}")
print(f"Evidence: {enhanced_corpus[top_idx]['evidence_level']}")
print(f"Severity: {enhanced_corpus[top_idx]['severity']}")

Test query: 'oily skin acne summer'

Top match:
Text: Acne vulgaris treatment includes topical retinoids (adapalene, tretinoin), benzoyl peroxide (2.5-10%...
Source: American Academy of Dermatology Acne Guidelines, 2024
Evidence: clinical_guideline
Severity: moderate


In [69]:
# Test the retrieval with different queries
test_queries = [
    "oily skin summer acne",  # Simulates NER entities
    "dry patches cheeks winter",
    "itchy neck perfume"
]

print("="*60)
print("TESTING RETRIEVAL")
print("="*60)

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    print("-" * 60)

    retrieved_docs, scores = retrieve_docs(query, top_k=3)

    for i, (doc, score) in enumerate(zip(retrieved_docs, scores), 1):
        print(f"\n{i}. [Relevance: {score:.3f}]")
        print(f"   {doc}")

    print()

TESTING RETRIEVAL

🔍 Query: 'oily skin summer acne'
------------------------------------------------------------

1. [Relevance: 0.561]
   Acne vulgaris treatment includes topical retinoids (adapalene, tretinoin), benzoyl peroxide (2.5-10%), or salicylic acid (0.5-2%). Avoid picking lesions to prevent scarring. For persistent acne lasting over 3 months, consult a dermatologist.

2. [Relevance: 0.509]
   Post-inflammatory hyperpigmentation (dark spots from healed acne) fades with ingredients like vitamin C (10-20%), niacinamide (4-5%), alpha arbutin (2%), and tranexamic acid (2-5%). Always use SPF 30+ to prevent darkening.

3. [Relevance: 0.501]
   For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores.


🔍 Query: 'dry patches cheeks winter'
------------------------------------------------------------

1. [Relevance: 0.549]
   Winter weather disrupts the s

In [70]:
# Function to extract entities from NER predictions and use them for retrieval
def ner_to_retrieval(sentence, predicted_tags, idx2word, idx2tag):
    """
    Extract entities from NER output and create query for retrieval

    Args:
        sentence: Tensor of word indices [seq_len]
        predicted_tags: Tensor of predicted tag indices [seq_len]
        idx2word: Dictionary mapping word index to word string
        idx2tag: Dictionary mapping tag index to tag string

    Returns:
        entities: List of extracted entity texts
        query: Combined query string for retrieval
    """
    entities = []
    current_entity = []

    for word_idx, tag_idx in zip(sentence, predicted_tags):
        # Skip padding
        if word_idx.item() == 0:
            continue

        word = idx2word[word_idx.item()]
        tag = idx2tag[tag_idx.item()]

        # If it's an entity tag (not 'O'), collect it
        if tag != 'O':
            current_entity.append(word)
        else:
            # If we were building an entity, save it
            if current_entity:
                entities.append(' '.join(current_entity))
                current_entity = []

    # Don't forget last entity
    if current_entity:
        entities.append(' '.join(current_entity))

    # Create query from entities
    query = ' '.join(entities)

    return entities, query

# Test function (you'll use this after NER prediction)
print("✓ NER-to-Retrieval connector ready!")
print("\nThis function will:")
print("1. Take your NER model's predictions")
print("2. Extract the entity phrases (skin types, symptoms, etc.)")
print("3. Create a search query from those entities")
print("4. Use that query to find relevant advice")

✓ NER-to-Retrieval connector ready!

This function will:
1. Take your NER model's predictions
2. Extract the entity phrases (skin types, symptoms, etc.)
3. Create a search query from those entities
4. Use that query to find relevant advice


In [71]:
# Simulate what happens after NER prediction
# (You'll replace this with actual NER model output later)

print("="*60)
print("SIMULATED END-TO-END: NER → RETRIEVAL → ADVICE")
print("="*60)

# Simulate a test case
test_sentence = "I have oily skin and get breakouts during summer"
simulated_entities = ["oily skin", "breakouts", "summer"]  # These would come from your NER model

print(f"\n📝 User Query: '{test_sentence}'")
print(f"🏷️  NER Extracted: {simulated_entities}")

# Create query from entities
query = ' '.join(simulated_entities)
print(f"🔍 Search Query: '{query}'")

# Retrieve relevant advice
retrieved_docs, scores = retrieve_docs(query, top_k=3)

print(f"\n📚 Retrieved {len(retrieved_docs)} relevant advice entries:")
print("-" * 60)

for i, (doc, score) in enumerate(zip(retrieved_docs, scores), 1):
    print(f"\n{i}. [Relevance: {score:.3f}]")
    print(f"   {doc}")

print("\n" + "="*60)
print("✓ End-to-end pipeline working!")

SIMULATED END-TO-END: NER → RETRIEVAL → ADVICE

📝 User Query: 'I have oily skin and get breakouts during summer'
🏷️  NER Extracted: ['oily skin', 'breakouts', 'summer']
🔍 Search Query: 'oily skin breakouts summer'

📚 Retrieved 3 relevant advice entries:
------------------------------------------------------------

1. [Relevance: 0.506]
   For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores.

2. [Relevance: 0.505]
   Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils.

3. [Relevance: 0.491]
   Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks. Use blotting papers for midday oil control.



In [78]:
from transformers import pipeline

# Load text generation model
print("Loading text generation model...")
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0 if torch.cuda.is_available() else -1
)
print("✓ Generator loaded!")

# Function to generate personalized advice
def generate_advice_with_citations(user_query, entities, retrieved_docs):
    """
    Generate advice that cites sources using [Source N] format

    Args:
        user_query: Original user question
        entities: List of extracted entities
        retrieved_docs: List of retrieved document dictionaries (from retrieve_docs_with_citations)

    Returns:
        response: Generated advice with citations
    """
    # Build context with source citations
    context = ""
    for i, doc in enumerate(retrieved_docs, 1):
        context += f"[Source {i}]: {doc['text']}\n"
        context += f"Citation: {doc['source']} (Evidence: {doc['evidence_level']})\n\n"

    # Create prompt that instructs model to cite sources
    prompt = f"""You are a knowledgeable skincare advisor. Provide advice using ONLY the information from the sources below.
Always cite sources using [Source N] format when making recommendations.
Be specific and practical.

Sources:
{context}

User concern: {user_query}
Key issues identified: {', '.join(entities) if entities else user_query}

Provide clear, evidence-based advice citing the sources above:"""

    # Generate response
    response = generator(
        prompt,
        max_length=250,
        min_length=60,
        do_sample=True,
        temperature=0.7
    )[0]['generated_text']

    return response

print("✓ Updated generation function with citation support")

Loading text generation model...


Device set to use cpu


✓ Generator loaded!
✓ Updated generation function with citation support


In [88]:
# Function to run complete NER → RAG pipeline
def complete_ner_rag_pipeline_final(user_sentence, model, word2idx, idx2word, idx2tag, max_len=25, confidence_threshold=0.5):
    """
    FINAL COMPLETE PIPELINE: NER → Confidence → Severity → Retrieval → Generation

    Args:
        user_sentence: Raw user input
        model: Trained NER model
        word2idx, idx2word, idx2tag: Vocabulary mappings
        max_len: Max sequence length
        confidence_threshold: Minimum confidence for retrieval

    Returns:
        Dictionary with all results including severity warnings
    """
    print("="*70)
    print("COMPLETE PIPELINE: NER → CONFIDENCE → SEVERITY → ADVICE")
    print("="*70)
    print(f"\n📝 User Input: {user_sentence}")

    # Step 1: Tokenize
    words = user_sentence.lower().split()
    word_indices = [word2idx.get(word, word2idx.get('<UNK>', 1)) for word in words]

    if len(word_indices) < max_len:
        word_indices = word_indices + [0] * (max_len - len(word_indices))
    else:
        word_indices = word_indices[:max_len]

    sentence_tensor = torch.LongTensor(word_indices).unsqueeze(0)

    # Step 2: NER Prediction
    model.eval()
    with torch.no_grad():
        tag_scores = model(sentence_tensor)
        predictions = torch.argmax(tag_scores, dim=-1).squeeze(0)

    # Step 3: Extract entities
    entities = []
    current_entity = []

    for i, (word_idx, tag_idx) in enumerate(zip(word_indices, predictions)):
        if word_idx == 0:
            break

        word = idx2word[word_idx]
        tag = idx2tag[tag_idx.item()]

        if tag != 'O' and not tag.startswith('I-'):
            if current_entity:
                entities.append(' '.join(current_entity))
            current_entity = [word]
        elif tag.startswith('I-') and current_entity:
            current_entity.append(word)
        else:
            if current_entity:
                entities.append(' '.join(current_entity))
                current_entity = []

    if current_entity:
        entities.append(' '.join(current_entity))

    print(f"🏷️  NER Extracted: {entities}")

    # Step 4: Retrieve with confidence check
    if entities:
        query = ' '.join(entities)
    else:
        query = user_sentence

    retrieved_docs, scores, confidence_status, low_confidence_msg = retrieve_with_confidence_check(
        query, top_k=3, confidence_threshold=confidence_threshold
    )

    # Handle low confidence
    if confidence_status == "low":
        print(f"\n⚠️  CONFIDENCE: LOW (Max score: {max(scores):.3f})")
        print(f"❌ Unable to provide reliable advice")

        return {
            'entities': entities,
            'advice': low_confidence_msg,
            'retrieved_docs': None,
            'scores': scores,
            'confidence': confidence_status,
            'severity': 'unknown',
            'warning': None
        }

    # Step 5: SEVERITY DETECTION (NEW!)
    severity_level, severity_warning, should_proceed = detect_severity(
        user_sentence, entities, retrieved_docs
    )

    print(f"\n🔍 SEVERITY CHECK: {severity_level.upper()}")

    if severity_warning:
        print(f"\n{severity_warning}")

    # If urgent, stop here and don't provide general advice
    if severity_level == "urgent":
        print("\n❌ Stopping: Urgent medical attention needed")
        print("="*70)

        return {
            'entities': entities,
            'advice': severity_warning,
            'retrieved_docs': retrieved_docs,
            'scores': scores,
            'confidence': confidence_status,
            'severity': severity_level,
            'warning': severity_warning
        }

    # Display confidence and sources
    if confidence_status == "high":
        print(f"\n✅ CONFIDENCE: HIGH (Max score: {max(scores):.3f})")
    else:
        print(f"\n⚡ CONFIDENCE: MEDIUM (Max score: {max(scores):.3f})")

    print(f"\n📚 Retrieved {len(retrieved_docs)} Sources:")
    for i, (doc, score) in enumerate(zip(retrieved_docs, scores), 1):
        print(f"\n   [Source {i}] - Relevance: {score:.3f}")
        print(f"   Text: {doc['text'][:80]}...")
        print(f"   Citation: {doc['source']}")
        if 'warning' in doc:
            print(f"   ⚠️  {doc['warning']}")

    # Step 6: Generate advice
    print(f"\n💡 Generating Advice...")
    advice = generate_advice_with_citations(user_sentence, entities, retrieved_docs)

    # Prepend severity warning if exists
    if severity_warning:
        full_advice = f"{severity_warning}\n\n{'─'*70}\n\nGENERAL INFORMATION:\n\n{advice}"
    else:
        full_advice = advice

    print(f"\n✨ FINAL ADVICE:")
    print(f"   {full_advice}")
    print("="*70)

    return {
        'entities': entities,
        'advice': full_advice,
        'retrieved_docs': retrieved_docs,
        'scores': scores,
        'confidence': confidence_status,
        'severity': severity_level,
        'warning': severity_warning
    }

print("✓ Final pipeline with severity detection ready")

✓ Final pipeline with severity detection ready


In [89]:
# IS CELL MEI USER QUERY KO CHANGE KRKE RESULTS KO DEKHTE RAHO KESE ARAHE HAIN
# Simple wrapper for easy use
def get_skincare_advice_final(user_input, confidence_threshold=0.5):
    """
    FINAL wrapper function with all features

    Args:
        user_input: User's skincare concern
        confidence_threshold: Minimum confidence (default 0.5)

    Returns:
        Complete results with advice, confidence, severity, warnings
    """
    result = complete_ner_rag_pipeline_final(
        user_input,
        model,
        word2idx,
        idx2word,
        idx2tag,
        max_len=25,
        confidence_threshold=confidence_threshold
    )

    return result

print("✓ Final wrapper function ready")

✓ Final wrapper function ready


# **ALL FUNCTION TESTING**

In [73]:
# Test the complete RAG system
print("="*70)
print("TESTING COMPLETE RAG PIPELINE: RETRIEVE + GENERATE")
print("="*70)

# Test cases
test_cases = [
    {
        "query": "I have oily skin and get breakouts during summer",
        "entities": ["oily skin", "breakouts", "summer"]
    },
    {
        "query": "My neck gets itchy after using perfumed lotion",
        "entities": ["neck", "itchy", "perfumed lotion"]
    },
    {
        "query": "I notice dry patches on my cheeks in winter",
        "entities": ["dry patches", "cheeks", "winter"]
    }
]

for idx, test in enumerate(test_cases, 1):
    print(f"\n{'='*70}")
    print(f"TEST CASE {idx}")
    print('='*70)

    user_query = test["query"]
    entities = test["entities"]

    print(f"📝 User Query: {user_query}")
    print(f"🏷️  Entities: {entities}")

    # Step 1: Retrieve relevant documents
    query = ' '.join(entities)
    retrieved_docs, scores = retrieve_docs(query, top_k=3)

    print(f"\n📚 Retrieved Documents:")
    for i, (doc, score) in enumerate(zip(retrieved_docs, scores), 1):
        print(f"   {i}. [Score: {score:.3f}] {doc[:80]}...")

    # Step 2: Generate personalized advice
    print(f"\n💡 Generated Advice:")
    advice = generate_advice(user_query, entities, retrieved_docs)
    print(f"   {advice}")
    print()

print("="*70)
print("✓ Complete RAG pipeline working!")
print("="*70)

TESTING COMPLETE RAG PIPELINE: RETRIEVE + GENERATE

TEST CASE 1
📝 User Query: I have oily skin and get breakouts during summer
🏷️  Entities: ['oily skin', 'breakouts', 'summer']

📚 Retrieved Documents:
   1. [Score: 0.506] For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for ...
   2. [Score: 0.505] Winter weather disrupts the skin barrier causing transepidermal water loss. Comb...
   3. [Score: 0.491] Combination skin requires zone-specific care: lightweight gel moisturizers on th...

💡 Generated Advice:


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores. Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils. Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks. Use blotting papers for midday oil control.


TEST CASE 2
📝 User Query: My neck gets itchy after using perfumed lotion
🏷️  Entities: ['neck', 'itchy', 'perfumed lotion']

📚 Retrieved Documents:
   1. [Score: 0.569] Contact dermatitis from fragranced products presents as itching, redness, and so...
   2. [Score: 0.468] Eczema (atopic dermatitis) maintenance involves fragrance-free moisturizers appl...
   3. [Score: 0.433] Sensitive 

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   My neck gets itchy after using perfumed lotion. Discontinue the product immediately and switch to fragrance-free alternatives. Eczema (atopic dermatitis) maintenance involves fragrance-free moisturizers applied 2-3 times daily, gentle cleansers, and identifying triggers. Severe itching, weeping, or infected lesions require immediate medical attention. Avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfactants like SLS. Choose products labeled hypoallergenic and fragrance-free.


TEST CASE 3
📝 User Query: I notice dry patches on my cheeks in winter
🏷️  Entities: ['dry patches', 'cheeks', 'winter']

📚 Retrieved Documents:
   1. [Score: 0.549] Winter weather disrupts the skin barrier causing transepidermal water loss. Comb...
   2. [Score: 0.531] Combination skin requires zone-specific care: lightweight gel moisturizers on th...
   3. [Score: 0.506] Dry skin requires moisturizers with humectants (hyaluronic acid, glycerin) and o...

💡 Generated Advice:


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to treat dry skin. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to treat dry skin. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to treat dry skin. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to treat dry skin. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to treat dry skin. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrol

✓ Complete RAG pipeline working!


In [75]:
# Test with your trained model
# Make sure you have: model, word2idx, idx2word, idx2tag loaded

test_sentences = [
    "I have oily skin and get pimples during summer",
    "My cheeks feel dry and flaky in winter",
    "I get itchy rashes after using perfumed products"
]

print("\n" + "="*70)
print("TESTING WITH YOUR TRAINED NER MODEL")
print("="*70 + "\n")

for sentence in test_sentences:
    entities, advice, sources = complete_ner_rag_pipeline(
        sentence,
        model,           # Your trained BiLSTM model
        word2idx,
        idx2word,
        idx2tag,
        max_len=25       # Adjust to your training max_len
    )
    print("\n" + "-"*70 + "\n")


TESTING WITH YOUR TRAINED NER MODEL

COMPLETE NER → RAG PIPELINE

📝 User Input: I have oily skin and get pimples during summer
🏷️  NER Extracted Entities: ['oily skin', 'pimples', 'summer']

📚 Retrieved 3 Relevant Documents:
   1. [Relevance: 0.525]
      For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing ...
   2. [Relevance: 0.499]
      Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (foreh...
   3. [Relevance: 0.474]
      Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick e...

💡 Generating Personalized Advice...


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores. For combination skin, use lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks. Avoid winter weather disrupting the skin barrier causing transepidermal water loss. Combat this with thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils.

----------------------------------------------------------------------

COMPLETE NER → RAG PIPELINE

📝 User Input: My cheeks feel dry and flaky in winter
🏷️  NER Extracted Entities: ['cheeks', 'dry', 'flaky', 'winter']

📚 Retrieved 3 Relevant Documents:
   1. [Relevance: 0.541]
      Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick e...
   2. [Relevance: 0.461]
      Cold-weat

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to keep skin moist during the cold season. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to keep skin moist during the cold season. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to keep skin moist during the cold season. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to keep skin moist during the cold season. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum) to keep skin moist during the cold season. Use moisturizers with humectants

----------------------------------------------------------------------

COMPLETE NER → RAG PIPELINE

📝 User Input: I get itchy rashes after using perfumed products
🏷️  NER Extracted Entities: ['itchy rashes', 'perfumed products']



Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   I get itchy rashes after using perfumed products. Discontinue the product immediately and switch to fragrance-free alternatives. Eczema (atopic dermatitis) maintenance involves fragrance-free moisturizers applied 2-3 times daily, gentle cleansers, and identifying triggers. Severe itching, weeping, or infected lesions require immediate medical attention. Avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfactants like SLS. Choose products labeled hypoallergenic and fragrance-free.

----------------------------------------------------------------------



In [90]:
# Test severity detection with various cases
print("\n" + "="*70)
print("TESTING SEVERITY DETECTION")
print("="*70 + "\n")

test_cases = [
    {
        "query": "I have oily skin and breakouts",
        "expected": "MILD - common concern"
    },
    {
        "query": "I have cystic acne on my jawline",
        "expected": "REQUIRES MEDICAL - cystic acne needs treatment"
    },
    {
        "query": "My face is bleeding and has severe pain",
        "expected": "URGENT - immediate attention needed"
    },
    {
        "query": "I get dry patches in winter",
        "expected": "MILD - seasonal concern"
    },
    {
        "query": "My skin is infected and has pus",
        "expected": "URGENT - infection warning"
    }
]

for idx, test in enumerate(test_cases, 1):
    print(f"\n{'='*70}")
    print(f"TEST {idx}: {test['expected']}")
    print('='*70)
    print(f"Query: {test['query']}")

    result = get_skincare_advice_final(test['query'])

    print(f"\n📊 RESULTS:")
    print(f"   Severity: {result['severity'].upper()}")
    print(f"   Confidence: {result['confidence'].upper()}")

    if result['warning']:
        print(f"   ⚠️  Warning issued: YES")
    else:
        print(f"   ✓ No special warnings")

    print("\n" + "-"*70)


TESTING SEVERITY DETECTION


TEST 1: MILD - common concern
Query: I have oily skin and breakouts
COMPLETE PIPELINE: NER → CONFIDENCE → SEVERITY → ADVICE

📝 User Input: I have oily skin and breakouts
🏷️  NER Extracted: ['oily skin', 'breakouts']

🔍 SEVERITY CHECK: REQUIRES_MEDICAL

⚠️ PROFESSIONAL CONSULTATION RECOMMENDED

Based on your concern, this may require evaluation by a dermatologist.

This condition requires prescription treatment

I can provide general skincare information, but professional diagnosis is recommended for best outcomes.

⚡ CONFIDENCE: MEDIUM (Max score: 0.522)

📚 Retrieved 3 Sources:

   [Source 1] - Relevance: 0.522
   Text: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for ...
   Citation: American Academy of Dermatology Guidelines, 2023

   [Source 2] - Relevance: 0.493
   Text: Combination skin requires zone-specific care: lightweight gel moisturizers on th...
   Citation: Clinical, Cosmetic and Investigational Dermatology, 2023

Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   ⚠️ PROFESSIONAL CONSULTATION RECOMMENDED

Based on your concern, this may require evaluation by a dermatologist.

This condition requires prescription treatment

I can provide general skincare information, but professional diagnosis is recommended for best outcomes.

──────────────────────────────────────────────────────────────────────

GENERAL INFORMATION:

[Source 1]: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores. [Source 2]: Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks. [Source 3]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 4]: Perioral dermatitis presents as red bumps around the mouth and nose. [Source 5]: Per

Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   ⚠️ MEDICAL EVALUATION NEEDED

Condition mentioned: cystic acne

This condition typically requires professional diagnosis and prescription treatment.

RECOMMENDED ACTION:
• Schedule appointment with a dermatologist
• Get proper diagnosis and treatment plan
• General skincare advice below may help with maintenance, but is not a substitute for medical care

I can provide general information, but professional evaluation is important.

──────────────────────────────────────────────────────────────────────

GENERAL INFORMATION:

Use only the information from the sources below. Use only the information from [Source N] format when making recommendations. Use only the information from [Source N] format when making recommendations. Use only the information from [Source N] format when making recommendations. Use only the information from [Source N] format when making recommendations. Use only the information from [Source N] format when making recommendations. Use only the info

Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   Use only the information from the sources below. Be specific and practical. Avoid harsh cleansers and over-exfoliation. Use thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils. Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Use moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum).

📊 RESULTS:
   Severity: MODERATE
   Confidence: MEDIUM
   ✓ No special warnings

----------------------------------------------------------------------

TEST 5: URGENT - infection warning
Query: My skin is infected and ha

In [85]:
# Test confidence scoring with various queries
print("\n" + "="*70)
print("TESTING CONFIDENCE SCORING")
print("="*70 + "\n")

test_cases = [
    {
        "query": "I have oily skin and breakouts in summer",
        "expected": "HIGH - matches corpus well"
    },
    {
        "query": "My neck itches after perfume",
        "expected": "HIGH - fragrance sensitivity in corpus"
    },
    {
        "query": "I have purple spots on my toenails",
        "expected": "LOW - not in corpus"
    },
    {
        "query": "What about laser treatment for scars",
        "expected": "LOW - medical procedure not in corpus"
    }
]

for idx, test in enumerate(test_cases, 1):
    print(f"\n{'='*70}")
    print(f"TEST CASE {idx}: {test['expected']}")
    print('='*70)

    result = get_skincare_advice_with_confidence(test['query'])

    print(f"\n📊 CONFIDENCE LEVEL: {result['confidence'].upper()}")

    if result['confidence'] == "low":
        print("✓ Correctly declined to answer - low confidence")
    else:
        print(f"✓ Retrieved {len(result['retrieved_docs'])} sources")
        print(f"✓ Generated advice with citations")

    print("\n" + "-"*70)


TESTING CONFIDENCE SCORING


TEST CASE 1: HIGH - matches corpus well
NER → RAG PIPELINE (WITH CONFIDENCE SCORING)

📝 User Input: I have oily skin and breakouts in summer
🏷️  NER Extracted: ['oily skin', 'breakouts', 'summer']

⚡ CONFIDENCE: MEDIUM (Max score: 0.506)

📚 Retrieved 3 Sources:

   [Source 1] - Relevance: 0.506
   Text: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for ...
   Citation: American Academy of Dermatology Guidelines, 2023

   [Source 2] - Relevance: 0.505
   Text: Winter weather disrupts the skin barrier causing transepidermal water loss. Comb...
   Citation: International Journal of Cosmetic Science, 2023

   [Source 3] - Relevance: 0.491
   Text: Combination skin requires zone-specific care: lightweight gel moisturizers on th...
   Citation: Clinical, Cosmetic and Investigational Dermatology, 2023

💡 Generating Advice...


Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   [Source 1]: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores. Citation: American Academy of Dermatology Guidelines, 2023 (Evidence: clinical_guideline) [Source 2]: Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils. Citation: International Journal of Cosmetic Science, 2023 (Evidence: peer_reviewed) [Source 3]: Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks. Use blotting papers for midday oil control.

📊 CONFIDENCE LEVEL: MEDIUM
✓ Retrieved 3 sources
✓ Generated advice with citations

----------------------------------------------------------------------

TEST CA

In [86]:
# Test different confidence thresholds
test_query = "I get dry patches on my cheeks"

print("\n" + "="*70)
print("TESTING DIFFERENT CONFIDENCE THRESHOLDS")
print("="*70)
print(f"\nQuery: '{test_query}'")

thresholds = [0.3, 0.5, 0.7]

for threshold in thresholds:
    print(f"\n{'─'*70}")
    print(f"Threshold: {threshold}")
    print('─'*70)

    result = get_skincare_advice_with_confidence(test_query, confidence_threshold=threshold)

    print(f"Confidence Status: {result['confidence']}")
    print(f"Max Score: {max(result['scores']):.3f}")

    if result['confidence'] == "low":
        print("❌ DECLINED - Confidence too low")
    else:
        print(f"✅ ACCEPTED - Generated advice")

print("\n" + "="*70)
print("Recommendation: Use threshold 0.5 for balanced performance")
print("="*70)


TESTING DIFFERENT CONFIDENCE THRESHOLDS

Query: 'I get dry patches on my cheeks'

──────────────────────────────────────────────────────────────────────
Threshold: 0.3
──────────────────────────────────────────────────────────────────────
NER → RAG PIPELINE (WITH CONFIDENCE SCORING)

📝 User Input: I get dry patches on my cheeks
🏷️  NER Extracted: ['dry', 'patches', 'cheeks']

⚡ CONFIDENCE: MEDIUM (Max score: 0.567)

📚 Retrieved 3 Sources:

   [Source 1] - Relevance: 0.567
   Text: Combination skin requires zone-specific care: lightweight gel moisturizers on th...
   Citation: Clinical, Cosmetic and Investigational Dermatology, 2023

   [Source 2] - Relevance: 0.521
   Text: Dry skin requires moisturizers with humectants (hyaluronic acid, glycerin) and o...
   Citation: Journal of Clinical and Aesthetic Dermatology, 2022

   [Source 3] - Relevance: 0.515
   Text: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using ...
   Citation: Dermatology Online Journal,

Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   Use only the information from the sources below. Use [Source N] format when making recommendations. Be specific and practical. [Source 1]: Dry skin requires moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Apply within 3 minutes after washing while skin is still damp to lock in moisture. [Source 2]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 3]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 4]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 5]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 6]:

Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE:
   You are a knowledgeable skincare advisor. Use only the information from the sources below. Always cite sources using [Source N] format when making recommendations. Be specific and practical. [Source 1]: Dry skin requires moisturizers with humectants (hyaluronic acid, glycerin) and occlusives (ceramides, petrolatum). Apply within 3 minutes after washing while skin is still damp to lock in moisture. [Source 2]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 3]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 4]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical steroids, fluorinated toothpaste, and heavy moisturizers. [Source 5]: Perioral dermatitis presents as red bumps around the mouth and nose. Stop using topical stero

In [81]:
# Test with citations
print("\n" + "="*70)
print("TESTING CITATIONS FEATURE")
print("="*70 + "\n")

test_cases = [
    "I have oily skin and get breakouts during summer",
    "My neck gets itchy after using perfumed lotion",
]

for idx, test_sentence in enumerate(test_cases, 1):
    print(f"\n{'='*70}")
    print(f"TEST CASE {idx}")
    print('='*70)

    result = get_skincare_advice_cited(test_sentence)

    # Display sources clearly
    print("\n📖 SOURCES USED:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        print(f"\n[Source {i}]")
        print(f"Text: {doc['text'][:100]}...")
        print(f"Citation: {doc['source']}")
        print(f"Evidence Level: {doc['evidence_level']}")
        if 'warning' in doc:
            print(f"⚠️  Warning: {doc['warning']}")

    print("\n" + "-"*70)


TESTING CITATIONS FEATURE


TEST CASE 1
COMPLETE NER → RAG PIPELINE (WITH CITATIONS)

📝 User Input: I have oily skin and get breakouts during summer
🏷️  NER Extracted Entities: ['oily skin', 'breakouts', 'summer']

📚 Retrieved 3 Relevant Sources:

   [Source 1] - Relevance: 0.506
   Text: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products c...
   Citation: American Academy of Dermatology Guidelines, 2023
   Evidence: clinical_guideline

   [Source 2] - Relevance: 0.505
   Text: Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this wi...
   Citation: International Journal of Cosmetic Science, 2023
   Evidence: peer_reviewed

   [Source 3] - Relevance: 0.491
   Text: Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-z...
   Citation: Clinical, Cosmetic and Investigational Dermatology, 2023
   Evidence: expert_consensus

💡 Generating Cited Advice...


Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE (WITH CITATIONS):
   [Source 1]: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing salicylic acid (0.5-2%) which helps control sebum production and prevents clogged pores. Citation: American Academy of Dermatology Guidelines, 2023 (Evidence: clinical_guideline) [Source 2]: Winter weather disrupts the skin barrier causing transepidermal water loss. Combat this with thick emollient moisturizers containing ceramides, use a humidifier indoors, and avoid very hot water which strips natural oils. Citation: International Journal of Cosmetic Science, 2023 (Evidence: peer_reviewed) [Source 3]: Combination skin requires zone-specific care: lightweight gel moisturizers on the oily T-zone (forehead, nose, chin), and richer creams on dry cheeks.

📖 SOURCES USED:

[Source 1]
Text: For oily skin, use a gentle foaming or gel-based cleanser twice daily. Look for products containing ...
Citation: American Academy of Dermatology Guidelines

Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✨ FINAL ADVICE (WITH CITATIONS):
   [Source 1]: Discontinue the product immediately and switch to fragrance-free alternatives. If symptoms persist beyond 48 hours, see a dermatologist. [Source 2]: Eczema (atopic dermatitis) maintenance involves fragrance-free moisturizers applied 2-3 times daily, gentle cleansers, and identifying triggers. Severe itching, weeping, or infected lesions require immediate medical attention. [Source 3]: Sensitive skin should avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfactants like SLS. Choose products labeled hypoallergenic and fragrance-free. [Source 4]: Sensitive skin should avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfactants like SLS. [Source 5]: Sensitive skin should avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfactants like SLS. [Source 6]: Sensitive skin should avoid common irritants: fragrances, essential oils, alcohol denat, and harsh surfac